In [5]:
import itertools
from pathlib import Path
from typing import Iterable, List, Tuple

import geopandas as gpd
import pandas as pd
from shapely.geometry import box
from shapely.ops import unary_union
from shapely.prepared import prep
from tqdm.auto import tqdm


In [6]:
INPUT_GPKG = Path("../../data/geo/regions/GTA_CSD.gpkg")
OUTPUT_DIR = Path("../../data/geo/geohash")
OUTPUT_GEOHASH6 = OUTPUT_DIR / "toronto_geohash6.csv"
OUTPUT_GEOHASH8 = OUTPUT_DIR / "toronto_geohash8.csv"

TARGET_CSD_NAME = "Toronto"
TARGET_CSD_FIELD = "CSDNAME"
GEODETIC_CRS = "EPSG:4326"

GEOHASH_BASE32 = "0123456789bcdefghjkmnpqrstuvwxyz"
ROOT_PREFIXES = list(GEOHASH_BASE32)


## Approach

This notebook avoids external geohash API calls and instead uses a recursive geohash-tree traversal:

1. Start from all 32 first-character geohash prefixes.
2. Convert each prefix to its cell bounding box.
3. Drop prefixes whose cell does **not** intersect Toronto.
4. If a cell is fully contained inside Toronto, expand all of its descendants directly without further geometry tests.
5. Otherwise subdivide only that prefix into its 32 children and continue until the requested precision.

This is efficient because it prunes large parts of the world immediately, and it still includes **all** border cells because the final inclusion test is based on `intersects`, not containment.


In [7]:
def geohash_bbox(geohash: str) -> Tuple[float, float, float, float]:
    """
    Return (min_lon, min_lat, max_lon, max_lat) for a geohash.
    This is a pure-Python decoder, so there is no external geohash API call.
    """
    lat_interval = [-90.0, 90.0]
    lon_interval = [-180.0, 180.0]
    is_even = True

    for char in geohash:
        cd = GEOHASH_BASE32.index(char)
        for mask in (16, 8, 4, 2, 1):
            if is_even:
                mid = (lon_interval[0] + lon_interval[1]) / 2
                if cd & mask:
                    lon_interval[0] = mid
                else:
                    lon_interval[1] = mid
            else:
                mid = (lat_interval[0] + lat_interval[1]) / 2
                if cd & mask:
                    lat_interval[0] = mid
                else:
                    lat_interval[1] = mid
            is_even = not is_even

    return lon_interval[0], lat_interval[0], lon_interval[1], lat_interval[1]


def geohash_polygon(geohash: str):
    min_lon, min_lat, max_lon, max_lat = geohash_bbox(geohash)
    return box(min_lon, min_lat, max_lon, max_lat)


def all_descendants(prefix: str, target_precision: int) -> Iterable[str]:
    """
    Yield every descendant geohash of `prefix` at `target_precision`.
    Used when a prefix cell is fully contained by Toronto, so no further
    geometry checks are needed.
    """
    remaining = target_precision - len(prefix)
    if remaining < 0:
        return
    if remaining == 0:
        yield prefix
        return

    for suffix_tuple in itertools.product(GEOHASH_BASE32, repeat=remaining):
        yield prefix + "".join(suffix_tuple)


In [8]:
def load_toronto_geometry(
    input_gpkg: Path,
    name_field: str = TARGET_CSD_FIELD,
    target_name: str = TARGET_CSD_NAME,
):
    """Read the GTA CSD file and return a valid Toronto geometry in EPSG:4326."""
    gdf = gpd.read_file(input_gpkg)

    if name_field not in gdf.columns:
        raise ValueError(
            f"Column {name_field!r} not found. Available columns: {list(gdf.columns)}"
        )

    toronto = gdf.loc[gdf[name_field] == target_name].copy()
    if toronto.empty:
        raise ValueError(
            f"No row found where {name_field!r} == {target_name!r}."
        )

    if toronto.crs is None:
        raise ValueError("Input GeoPackage has no CRS defined.")

    toronto = toronto.to_crs(GEODETIC_CRS)
    geom = unary_union(toronto.geometry)
    geom = geom.buffer(0)

    if geom.is_empty:
        raise ValueError("Toronto geometry is empty after loading/repair.")

    return geom


toronto_geom = load_toronto_geometry(INPUT_GPKG)
toronto_bounds = toronto_geom.bounds

print("Toronto bounds:", toronto_bounds)
print("Toronto geometry type:", toronto_geom.geom_type)


Toronto bounds: (-79.6393024056249, 43.57920109696593, -79.1153329419791, 43.855465496027506)
Toronto geometry type: MultiPolygon


In [9]:
def geohashes_intersecting_geometry(
    geometry,
    target_precision: int,
    *,
    show_progress: bool = True,
) -> List[str]:
    """
    Return every geohash of the requested precision whose cell intersects
    the input geometry.

    Border cells are included even if the overlap is tiny.
    """
    if target_precision < 1:
        raise ValueError("target_precision must be at least 1")

    prepared = prep(geometry)
    results: List[str] = []
    stack = ROOT_PREFIXES[::-1]

    pbar = tqdm(
        total=len(stack),
        disable=not show_progress,
        desc=f"precision {target_precision}",
        unit="cell",
    )
    pbar.set_postfix(found=0, queued=len(stack))

    while stack:
        prefix = stack.pop()
        cell = geohash_polygon(prefix)

        if not prepared.intersects(cell):
            pbar.update(1)
            if pbar.n % 5000 == 0:
                pbar.set_postfix(found=len(results), queued=len(stack))
            continue

        if len(prefix) == target_precision:
            results.append(prefix)
            pbar.update(1)
            if pbar.n % 5000 == 0:
                pbar.set_postfix(found=len(results), queued=len(stack))
            continue

        if prepared.contains(cell):
            for descendant in all_descendants(prefix, target_precision):
                results.append(descendant)
            pbar.update(1)
            if pbar.n % 5000 == 0:
                pbar.set_postfix(found=len(results), queued=len(stack))
            continue

        children = [prefix + ch for ch in reversed(GEOHASH_BASE32)]
        stack.extend(children)
        pbar.total = max(pbar.total, pbar.n + len(stack))
        pbar.update(1)

        if pbar.n % 5000 == 0:
            pbar.set_postfix(found=len(results), queued=len(stack))

    pbar.total = pbar.n
    pbar.set_postfix(found=len(results), queued=0)
    pbar.close()

    results.sort()
    return results


In [10]:
def write_geohash_csv(geohashes: List[str], out_csv: Path) -> None:
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame({"geohash": geohashes}).to_csv(out_csv, index=False)
    print(f"Wrote {len(geohashes):,} geohashes to {out_csv}")


def run_precision(target_precision: int, out_csv: Path) -> List[str]:
    geohashes = geohashes_intersecting_geometry(
        toronto_geom,
        target_precision=target_precision,
        show_progress=True,
    )
    write_geohash_csv(geohashes, out_csv)
    return geohashes


## Run precision 6

This will create `data/geo/geohash/toronto_geohash6.csv`.


In [11]:
geohash6 = run_precision(6, OUTPUT_GEOHASH6)
len(geohash6)


precision 6:   0%|          | 0/32 [00:00<?, ?cell/s]

Wrote 1,307 geohashes to ../../data/geo/geohash/toronto_geohash6.csv


1307

## Run precision 8

This will create `data/geo/geohash/toronto_geohash8.csv`.

Precision 8 is much larger than precision 6, so expect it to take longer and produce a much bigger CSV.


In [12]:
geohash8 = run_precision(8, OUTPUT_GEOHASH8)
len(geohash8)


precision 8:   0%|          | 0/32 [00:00<?, ?cell/s]

Wrote 1,210,820 geohashes to ../../data/geo/geohash/toronto_geohash8.csv


1210820

## Optional quick checks


In [13]:
pd.read_csv(OUTPUT_GEOHASH6).head(), pd.read_csv(OUTPUT_GEOHASH8).head()


(  geohash
 0  dpxrvv
 1  dpxrvy
 2  dpxrvz
 3  dpxryj
 4  dpxryn,
     geohash
 0  dpxrvvtz
 1  dpxrvvvb
 2  dpxrvvvc
 3  dpxrvvvf
 4  dpxrvvvg)

## Notes

- The output includes **every geohash cell that intersects Toronto**, including border cells where only a tiny fraction overlaps.
- The progress bar tracks the recursive cell traversal, not the final number of output rows.
- The main speed-up comes from skipping non-intersecting prefixes early and bulk-expanding cells that are fully inside Toronto.
- If you want to keep border cells, use `intersects`. Do not switch it to `within` or `contains`.
